In [ ]:
# Requires the following modifications to the OWR code
# BaseClasses.py:
# Add `self.text_access_rule = ""` to Entrance and Location Classes

# Rules.py:
# import dill
# import re

# def clean_text_rule(rule):
#     new_text = dill.source.getsource(rule).strip()
#     # Only care about the inside of the lambda
#     new_text = re.sub(r'.*lambda state: ', '', new_text)
#     # state.
#     new_text = re.sub(r'state.', '', new_text)
#     # ( player )
#     new_text = re.sub(r'\(player\)', '', new_text)
#     # "has('
#     new_text = re.sub(r"has\('", '', new_text)
#     # , player
#     new_text = re.sub(r", player\)", '', new_text)
#     new_text = re.sub(f"#.*", '', new_text)  # remove comments
#     new_text = re.sub(r"\n\s+", ' ', new_text)  # remove newlines, replace with a space
#     new_text = re.sub(r"'", '', new_text)  # remove newlines, replace with a space
#     # get rid of extra parentheses, quotes etc.
#     new_text = new_text.strip(")'\"")
#     return f"|{new_text}|"

# Add `spot.text_access_rule = clean_text_rule(rule)` to `set_rule`

# Replace add_rule
# def add_rule(spot, rule, combine='and'):
#     old_rule = spot.access_rule
#     old_text = spot.text_access_rule
#     new_text = clean_text_rule(rule)

#     if combine == 'or':
#         spot.access_rule = lambda state: rule(state) or old_rule(state)
#         spot.text_access_rule = f"({old_text}) or ({new_text})" if old_text else new_text
#     else:
#         spot.access_rule = lambda state: rule(state) and old_rule(state)
#         spot.text_access_rule = f"({old_text}) and ({new_text})" if old_text else new_text


In [1]:
from BaseClasses import World

In [2]:
from CLI import parse_cli, parse_settings
from Dungeons import dungeon_keys, dungeon_bigs
from Main import main
from OWEdges import OWTileRegions

from source.classes.BabelFish import BabelFish

args = parse_settings()
args = parse_cli({})

lang = "en"
fish = BabelFish(lang=lang)


In [3]:
args.pottery[1] = 'lottery'
args.dropshuffle[1] = 'underworld'
args.bonk_drops[1] = True

args.logic[1] = 'noglitches'
world: World = main(args=args, fish=fish, seed=0)
worldAllState = world.get_all_state(keys=False)

In [4]:
# args.logic[1] = 'owglitches'
# owgWorld: World = main(args=args, fish=fish)

args.logic[1] = 'noglitches'
args.mode[1] = 'inverted'
invertedWorld: World = main(args=args, fish=fish)
invertedAllState = invertedWorld.get_all_state(keys=False)

# args.logic[1] = 'owglitches'
# owgInvertedWorld: World = main(args=args, fish=fish)


In [6]:
from BaseClasses import CrystalBarrier

hasCrystalSwitch = set()
blueBarrier = set()
orangeBarrier = set()



for d in world.doors:
    if d.crystal == CrystalBarrier.Either:
        hasCrystalSwitch.add(d.name)
    elif d.crystal == CrystalBarrier.Blue:
        blueBarrier.add(d.name)
    elif d.crystal == CrystalBarrier.Orange:
        orangeBarrier.add(d.name)

In [7]:
import ast

In [8]:
def find_no_exit_doors(file_path):
    with open(file_path, "r") as f:
        tree = ast.parse(f.read())

    no_exit_doors = []

    class DoorVisitor(ast.NodeVisitor):
        def visit_Call(self, node):
            # Check if this call or any parent call in the chain is .no_exit()
            # and if the root is create_door
            
            # We want to find the whole chain
            if self.is_no_exit_call(node):
                door_name = self.get_create_door_name(node)
                if door_name:
                    no_exit_doors.append(door_name)
            
            self.generic_visit(node)

        def is_no_exit_call(self, node):
            # Check if .no_exit is anywhere in the call chain of node
            curr = node
            while isinstance(curr, ast.Call):
                if isinstance(curr.func, ast.Attribute):
                    if curr.func.attr == 'no_exit':
                        return True
                    curr = curr.func.value
                else:
                    break
            return False

        def get_create_door_name(self, node):
            # Find the create_door call at the base of the chain and return the door name
            curr = node
            while isinstance(curr, ast.Call):
                if isinstance(curr.func, ast.Name) and curr.func.id == 'create_door':
                    # First argument is player, second is door name
                    if len(curr.args) >= 2 and isinstance(curr.args[1], ast.Constant):
                        return curr.args[1].value
                    elif len(curr.args) >= 2 and isinstance(curr.args[1], ast.Str): # for older python
                        return curr.args[1].s
                    return None
                
                if isinstance(curr.func, ast.Attribute):
                    curr = curr.func.value
                else:
                    break
            return None

    visitor = DoorVisitor()
    visitor.visit(tree)
    return sorted(list(set(no_exit_doors)))

one_way_doors = find_no_exit_doors("Doors.py")


In [9]:
format_test_cases = [
    ("|False)  |", {"always": {"allOf": ["never"]}}),
    ("", {}),
    (
        "((|can_use_bombs|) and (|has_Pearl|)) and (|has_Pearl|)",
        {"always": {"allOf": ["bomb", "moonpearl"]}},
    ),
    ("|Hookshot|", {"always": {"allOf": ["hookshot"]}}),
    (
        "|can_shoot_arrows or Red Boomerang or Fire Rod or Ice Rod or Cane of Somaria) |",
        {"always": {"anyOf": ["bow", "boomerang", "firerod", "icerod", "somaria"]}},
    ),
    (
        "((|can_reach_blue(world.get_region(Mire Crystal Dead End|) or (|eval_alternative_crystal_main( door_name, dungeon|)) and (|has_Pearl|)",
        {"always": {"allOf": ["moonpearl"]}},
    ),
    (
        "(|can_reach_blue(world.get_region(TR Crystal Maze Start|) and (|has_Pearl|)",
        {"always": {"allOf": ["moonpearl"]}},
    ),
    (
        "|Mushroom and can_reach(Potion Shop Area, Region|",
        {"always": {"allOf": ["mushroom", "canReach|Potion Shop Area"]}},
    ),
    (
        "| | and |moonpearl|",
        {"always": {"allOf": ["moonpearl"]}},
    ),
    (
        "|Lamp or world.free_lamp_cone[player]|",
        {"always": {"allOf": ["lantern"]}},
    ),
    (
        "(|eval_small_key_door_partial_main( door_name, dungeon|) and (|Lamp or world.free_lamp_cone[player]|)",
        {"always": {"allOf": ["lantern"]}},
    )
]

enemy_format_test_cases = [
    (
        "(|rule1( or rule2(|) and (|has_Pearl|)",
        {"always": {"allOf": ["melee", "bomb", "moonpearl"]}},
        "StalfosKnight",
    ),
    ("|has(item, player, count|", {"always": {"allOf": ["hammer"]}}, "Terrorpin"),
]

mapping = {
    "can_use_bombs": "bomb",
    "can_shoot_arrows": "bow",
    "has_Pearl": "",
    "Red Boomerang": "boomerang",
    "Blue Boomerang": "boomerang",
    "Book of Mudora": "book",
    "Fire Rod": "firerod",
    "Ice Rod": "icerod",
    "Magic Powder": "powder",
    "Cane of Somaria": "somaria",
    "Cane of Byrna": "byrna",
    "Ocarina": "flute",
    "Lamp": "lantern",
    "green pendant": "greenPendant",
    "red pendant": "canPullPedestal",
    "blue pendant": "canPullPedestal",
    "has_sword": "sword",
    "tempered sword": "temperedSword",
    "golden sword": "goldSword",
    "has_boots": "boots",
    "has_mirror": "mirror",
    "Pegasus Boots": "boots",
    "has_bottle": "bottle",
    "silver arrows": "canUseSilverArrows",
    "escort old man": "canCollectOldMan",
    "return old man": "canRescueOldMan",
    "pick up kiki": "canCollectKiki",
    "dark palace opened": "canOpenPod",
    "get frog": "canCollectFrog",
    "return smith": "canRescueBlacksmith",
    "pick up purple chest": "canCollectPurpleChest",
    "deliver purple chest": "canDeliverPurpleChest",
    "pick up big bomb": "canCollectBigBomb",
    "detonate big bomb": "canOpenPyramidFairy",
    "crystal 5": "canBuyBigBomb",
    "crystal 6": "canBuyBigBomb",
    "world.is_pyramid_open": "isPyramidOpen",
    "can_kill_most_things": "canKillMostEnemies",
    "has_misery_mire_medallion": "medallion|mm",
    "has_turtle_rock_medallion": "medallion|tr",
    "turtle opened": "canOpenTR",
    "beat agahnim 2": "agahnim2",
    "can_reach_orange.*": "", #"orangeSwitchDown",
    "can_reach_blue.*": "", #"blueSwitchDown",
    "can_reach\((.*),.* ": r"canReach|\1",
    "can_flute": "canFlute",
    "can_extend_magic\(, (\d+).*": r"canExtendMagic|\1",
    "can_hit_crystal": "canHitSwitch",
    "can_hit_crystal_through_barrier": "canHitRangedSwitch",
    "can_lift_rocks": "glove",
    "can_lift_heavy_rocks": "mitts",
    "has_fire_source": "canLightFires",
    "can_collect_bonkdrops": "canGetBonkableItem",
    "can_avoid_lasers": "canAvoidLasers",
    "can_melt_things": "canBurnThings",
    "has_beaten_aga": "agahnim",
    "world.free_lamp_cone.*": "",
    "has_beam_sword": "swordbeams",
    "def hidden_pits_rule: return hidden pits": "",
    "entrance.parent_region.dungeon.*": "",
    "world.can_take_damage": "canTakeDamage",
    "has(item_name": "",
    "not trench 1 filled": "",
    "trench 1 filled": "",
    "not trench 2 filled": "",
    "trench 2 filled": "",
    "drained swamp": "",
    "has_crystals(world.crystals_needed_for_gt[player]": "canOpenGT",
    "has_crystals(world.crystals_needed_for_gt[]": "canOpenGT",
    "has_crystals(world.crystals_needed_for_ganon[]": "canBeatGanon",
    "set_rule(world.get_location(spike cave, lambda  hammer": "",  # Spike cave hammer is only picked up on certain things
    "not has_beaten_aga": "",
    "true": "",
    "rule1": "",
    "rule2": "",
    "item": "",
    "world.logic[] in [noglitches, minorglitches]": "",
    "has_hearts(.*": "",
    "open floodgate": "canReach|Dam",
    "shining light": "canOpenTTAttic",
    "maiden unmasked, entrance.": "canRevealBlind",
    "maiden rescued": "canRevealBlind",
    "world.get_region.*": "",
    "convenient block": "canReach|Ice Crystal Block",
    "def hidden_pits_rule(: return hidden pits": "",
    "hidden_pits_rule": "",
    "has_sm_key.*": "",
    "has_hearts.*": "",
    "agahnim, or": "",
    "boolean": "",
    "eval_small_key_door.*": "",
    "eval_alternative_crystal.*": "",
}

ow_mapping = mapping.copy()
ow_mapping.update(
    {
        "has_Pearl": "moonpearl",
    }
)

enemy_rules = {
    "Pokey": "canBurnThings",
    "RedBari": "canBurnThings",
    "RedEyegore": "bow",
    "Terrorpin": "hammer",
    "StalfosKnight": "melee and bomb",
}

import ast
import re


def format_rule(rule, mapping=mapping, isEnemy=""):
    """ "
    Rules are output as combinations of allOf and anyOf objects, with requirements being in lists.
    If not possible, return "never"
    If no rules, return {} (i.e. always true)
    Rules are always encapsulated in in a higher level object "always"
    i.e.:
        {
            "always": {
                "allOf": [
                    {
                        "anyOf": [
                            "pearl",
                            "flute"
                        ]
                    },
                    "hookshot"
                ]
            }
        }

    There are many edge cases to consider:
    - Some requirements will be empty strings - these should return as true (i.e. {})
    - All requirements are extracted extracted from lambda functions should be encapsulated in pipes `|requirement|`
    - Requirements that used the built in OR/AND functions have been encapsulated in brackets `(|requirement1| or |requirement2|)`/`(|requirement1| and |requirement2|)`
    - Some requirements have an " or " or " and " statement within them from the lambda text rules, these will _not_ be further encapsulated in parentheses because they come from the same rule
    - Some lambda text rules will have python syntax in them, such as:
      - `|has_crystals(world.crystals_needed_for_gt[player]|`
      - `world.is_pyramid_open`
      - `|has_crystals(world.crystals_needed_for_gt[player] or world.shuffle[player] in (restricted, full, lite, lean, district, swapped, crossed, insanity|`
    - Some requirements will have multiple words separated by spaces, such as:
      - `|Turtle Opened|`
      - `|world.can_take_damage or Cape or Cane of Byrna or has_sword|`
        - This is actually a larger or statement that should result in the following `(|world.can_take_damage| or |Cape| or |Cane of Byrna| or |has_sword|)` and then be converted to JSON appropriately
    - Some requirements are duplicated, e.g. `((|can_lift_rocks|) and (|has_Pearl|)) and (|has_Pearl|)`. It would be better to deduplicate these to `(|can_lift_rocks|) and (|has_Pearl|)`
    - Some requirements may not be determinable and will say `|boolean|`. Some other rules could not be extracted properly and will say `|rule1( or rule2(|`.
      - These should be able to be changed using an old -> new mapping provided externally
    - Some requirements may have data in them that is not the same each time, and we might need to use regex to extract the core requirement, i.e.
      - `"|can_reach_orange(world.get_region(PoD Arena North|"` -> `"|canHitSwitch|"`
    - In some cases, the isEnemy parameter is provided, and if the rule matches an enemy in the enemy_rules mapping, it should add the rule with that requirement.
      - Usually the python-esque code should be replaced with the enemy requirement as well.
    """
    if not rule:
        return {}

    # Pre-cleaning
    rule = rule.strip()

    # Create lowercased mapping, stripping keys
    lower_mapping = {k.lower().strip(): v for k, v in mapping.items()}

    # 1. Replace |...| with placeholders
    placeholders = {}
    counter = 0

    def sub_cb(match):
        nonlocal counter
        key = f"REQ_{counter}"
        placeholders[key] = match.group(1)
        counter += 1
        return key

    processed_rule = re.sub(r"\|([^|]+)\|", sub_cb, rule)

    # If processed_rule is empty or just whitespace, return {}
    if not processed_rule.strip():
        return {}

    # 2. Parse with AST
    try:
        tree = ast.parse(processed_rule, mode="eval")
    except SyntaxError:
        # Fallback for simple strings or malformed
        if "False" in rule:
            return {"always": {"allOf": ["never"]}}
        return {"always": {"allOf": ["never"]}}

    # 3. Recursive processor
    def parse_node(node):
        if isinstance(node, ast.BoolOp):
            op = "allOf" if isinstance(node.op, ast.And) else "anyOf"
            values = [parse_node(v) for v in node.values]
            return {op: values}
        elif isinstance(node, ast.Name):
            content = placeholders.get(node.id, "")
            return process_requirement(content)
        elif isinstance(node, ast.Expression):
            return parse_node(node.body)
        return {}

    def process_requirement(req_str):
        # Clean up
        req_str = req_str.strip()

        # Handle has(...)
        # has('Item')
        m = re.search(r"has\(['\"]([^'\"]+)['\"]\)", req_str)
        if m:
            req_str = m.group(1)
        # has(item, ...)
        m = re.search(r"has\(([^,]+),", req_str)
        if m:
            req_str = m.group(1)

        # Remove `lambda state:`, `state.`, `(player)`, `player`
        req_str = req_str.replace("lambda state:", "")
        req_str = req_str.replace("state.", "")
        req_str = req_str.replace("(player)", "")
        req_str = req_str.replace("player", "")

        # Remove trailing/leading parens/spaces
        req_str = req_str.strip("() ")

        # Do NOT lowercase here to preserve case for regex replacements

        return parse_simple_rule(req_str)

    def apply_mapping(req_str):
        # Handle isEnemy
        if isEnemy and isEnemy in enemy_rules:
            # Heuristic: if req_str contains "rule" or "boolean" or "item", replace with enemy rule
            # Check lowercased for heuristic
            lower_req = req_str.lower()
            if "rule" in lower_req or "boolean" in lower_req or "item" in lower_req:
                return parse_simple_rule(enemy_rules[isEnemy])

        # Apply mapping
        # First pass: exact match (case insensitive)
        if req_str.lower() in lower_mapping:
            res = lower_mapping[req_str.lower()]
            if res == "":
                return ""  # Return empty string to be filtered out, not {} which is "always true"
            return res

        # Second pass: partial replacement and regex
        sorted_keys = sorted(lower_mapping.keys(), key=len, reverse=True)
        replaced = False
        for k in sorted_keys:
            v = lower_mapping[k]
            # Try literal first (case insensitive)
            if k in req_str.lower():
                # Use re.sub with IGNORECASE to replace literal k
                # Escape k to treat it as literal
                try:
                    new_str = re.sub(re.escape(k), v, req_str, flags=re.IGNORECASE)
                    if new_str != req_str:
                        req_str = new_str
                        replaced = True
                except re.error:
                    pass
                continue

            # Try regex if it looks like one (contains *)
            if "*" in k or "\\" in k or "(" in k:
                try:
                    new_str = re.sub(k, v, req_str, flags=re.IGNORECASE)
                    if new_str != req_str:
                        req_str = new_str
                        replaced = True
                except re.error:
                    pass

        # Check if result is empty after replacements
        req_str = req_str.strip()
        if not req_str:
            return ""  # Return empty string to be filtered out in anyOf, not {} which short-circuits

        # Handle eval_ heuristic - treat as ignored since these are
        # dynamic checks that we handle elsewhere (like small key doors)
        if req_str.lower().startswith("eval_"):
            return ""

        # Handle "False"
        if "false" in req_str.lower():
            return "never"

        # If no replacement happened, lowercase it (default behavior)
        if not replaced:
            return req_str.lower()

        return req_str

    def parse_simple_rule(text):
        if " or " in text:
            parts = [parse_simple_rule(p.strip()) for p in text.split(" or ")]
            return {"anyOf": parts}
        if " and " in text:
            parts = [parse_simple_rule(p.strip()) for p in text.split(" and ")]
            return {"allOf": parts}

        text = text.strip()
        # Use strip instead of replace to preserve internal parens
        text = text.strip("() ")

        return apply_mapping(text)

    result = parse_node(tree)

    # 4. Simplify
    def simplify(obj):
        # if obj == "": return {} # Treat empty string as empty dict (True)

        if isinstance(obj, dict):
            if "allOf" in obj:
                items = [simplify(x) for x in obj["allOf"]]
                # Flatten
                new_items = []
                for i in items:
                    if isinstance(i, dict) and "allOf" in i:
                        new_items.extend(i["allOf"])
                    else:
                        new_items.append(i)
                # Filter empty
                new_items = [i for i in new_items if i != {} and i != ""]
                # Deduplicate
                unique = []
                for i in new_items:
                    if i not in unique:
                        unique.append(i)

                if "never" in unique:
                    return "never"
                if not unique:
                    return {}
                if len(unique) == 1:
                    return unique[0]
                return {"allOf": unique}

            if "anyOf" in obj:
                items = [simplify(x) for x in obj["anyOf"]]
                new_items = []
                for i in items:
                    if isinstance(i, dict) and "anyOf" in i:
                        new_items.extend(i["anyOf"])
                    else:
                        new_items.append(i)

                # Check for True ({})
                if any(i == {} for i in new_items):
                    return {}

                # Filter "" but track if we saw it
                has_empty_str = "" in new_items
                new_items = [i for i in new_items if i != ""]

                unique = []
                for i in new_items:
                    if i not in unique:
                        unique.append(i)

                if "never" in unique:
                    unique.remove("never")

                if not unique:
                    return "" if has_empty_str else "never"

                if len(unique) == 1:
                    return unique[0]
                return {"anyOf": unique}
        return obj

    final = simplify(result)

    # Special handling for enemy rules to match test expectation
    if isEnemy and isinstance(final, str) and final != "never" and final != "":
        final = {"allOf": [final]}

    # Ensure single items are wrapped in allOf
    if isinstance(final, str) and final != "":
        final = {"allOf": [final]}

    if final == "":
        return {}

    if not final:
        return {}

    return {"always": final}


# Run tests
print("Running tests...")
for rule, expected in format_test_cases:
    result = format_rule(rule, ow_mapping)
    if result != expected:
        print(f"FAILED: {rule}\nExpected: {expected}\nGot: {result}")
    else:
        print(f"PASSED: {rule} -> {result}")

for rule, expected, enemy in enemy_format_test_cases:
    result = format_rule(rule, ow_mapping, enemy)
    if result != expected:
        print(f"FAILED: {rule} (Enemy: {enemy})\nExpected: {expected}\nGot: {result}")
    else:
        print(f"PASSED: {rule} (Enemy: {enemy}) -> {result}")


def check_rule_keys(loc, state=worldAllState):
    # We need to check if keys are needed, this is easy for single entrance dungeons
    if loc.parent_region.dungeon:
        dungeon_name = loc.parent_region.dungeon.name
        sk = 0
        bk = 0
        for i in range(0, 12):
            stateCopy = state.copy()
            r = False
            stateCopy.prog_items[dungeon_keys[dungeon_name], 1] += 1
            for bk in range(0, 2):
                stateCopy.prog_items[dungeon_bigs[dungeon_name], 1] += 1
                if loc.access_rule(state):
                    r = True
                    if sk > 0 or bk > 0:
                        print(
                            "Location",
                            loc.name,
                            "in",
                            loc.parent_region.name,
                            "needs",
                            sk,
                            "small keys and",
                            bk,
                            "big keys",
                        )
                    break
            if r:
                if sk > 0 or bk > 0:
                    print(
                        "Location",
                        loc.name,
                        "in",
                        loc.parent_region.name,
                        "needs",
                        sk,
                        "small keys and",
                        bk,
                        "big keys",
                    )
                break
        # if not r:
        # print("Location", loc.name, "in", loc.parent_region.name, "needs more than 11 small keys and 1 big key")
    return format_rule(loc.text_access_rule, ow_mapping if loc.parent_region.type in [RegionType.LightWorld, RegionType.DarkWorld] else mapping)



Running tests...
PASSED: |False)  | -> {'always': {'allOf': ['never']}}
PASSED:  -> {}
PASSED: ((|can_use_bombs|) and (|has_Pearl|)) and (|has_Pearl|) -> {'always': {'allOf': ['bomb', 'moonpearl']}}
PASSED: |Hookshot| -> {'always': {'allOf': ['hookshot']}}
PASSED: |can_shoot_arrows or Red Boomerang or Fire Rod or Ice Rod or Cane of Somaria) | -> {'always': {'anyOf': ['bow', 'boomerang', 'firerod', 'icerod', 'somaria']}}
PASSED: ((|can_reach_blue(world.get_region(Mire Crystal Dead End|) or (|eval_alternative_crystal_main( door_name, dungeon|)) and (|has_Pearl|) -> {'always': {'allOf': ['moonpearl']}}
PASSED: (|can_reach_blue(world.get_region(TR Crystal Maze Start|) and (|has_Pearl|) -> {'always': {'allOf': ['moonpearl']}}
PASSED: |Mushroom and can_reach(Potion Shop Area, Region| -> {'always': {'allOf': ['mushroom', 'canReach|Potion Shop Area']}}
PASSED: | | and |moonpearl| -> {'always': {'allOf': ['moonpearl']}}
PASSED: |Lamp or world.free_lamp_cone[player]| -> {'always': {'allOf': ['la

In [10]:
ignored_tos = [
    "Flute Sky",
]

import json
from BaseClasses import RegionType

all_rules = set()

entrance_graph = {}

all_doors = {d.name: d for d in world.doors}
all_doors_inverted = {d.name: d for d in invertedWorld.doors}

# , "Inverted": invertedWorld
for region in world.regions:
    entrance_graph[region.name] = {
        "exits": {},
        "entrances": [],
        "type": region.type.name,
        "locations": {},
    }
    exits = entrance_graph[region.name]["exits"]
    entrances = entrance_graph[region.name]["entrances"]
    locations = entrance_graph[region.name]["locations"]
    reg = entrance_graph[region.name]
    if region.type in [RegionType.LightWorld, RegionType.DarkWorld]:
        reg['owid'] = OWTileRegions.get(region.name, None)
    else:
        for x in region.exits:
            door = all_doors.get(x.name, None)
            if door:
                reg['tileid'] = door.roomIndex
    for exit in region.exits:
        # Track entrances from OW to UW
        if exit.connected_region:
            if exit.name.startswith("Mirror From ") or exit.name.startswith(
                "Mirror To "
            ):
                # Add "has_mirror" requirement
                exit.text_access_rule = (
                    f"(|has_mirror|) and ({exit.text_access_rule})"
                    if exit.text_access_rule
                    else "|has_mirror|"
                )

            if exit.connected_region.name in ignored_tos:
                continue
            if exit.connected_region.type in [
                RegionType.Cave,
                RegionType.Dungeon,
            ] and region.type in [RegionType.LightWorld, RegionType.DarkWorld]:
                entrances.append(exit.name)
            exit_dict = {
                "to": exit.connected_region.name,
                "type": exit.connected_region.type.name,
                "requirements": {"Open": check_rule_keys(exit), "Inverted": "never"},
            }
            exits[exit.name] = exit_dict
            all_rules.add(json.dumps(check_rule_keys(exit)))
    for location in region.locations:
        locations[location.name] = {
            "requirements": {"Open": check_rule_keys(location), "Inverted": "never"}
        }
        all_rules.add(json.dumps(check_rule_keys(location)))
        if "Enemy" in location.name:
            locations[location.name]["type"] = location.note

In [11]:
for region in invertedWorld.regions:
    exits = entrance_graph[region.name]["exits"]
    entrances = entrance_graph[region.name]["entrances"]
    locations = entrance_graph[region.name]["locations"]
    reg = entrance_graph[region.name]
    if region.type in [RegionType.LightWorld, RegionType.DarkWorld]:
        reg['owid'] = OWTileRegions.get(region.name, None)
    else:
        for x in region.exits:
            door = all_doors_inverted.get(x.name, None)
            if door:
                reg['tileid'] = door.roomIndex
    for exit in region.exits:
        if exit.name.startswith("Mirror From ") or exit.name.startswith("Mirror To "):
            exit.text_access_rule = (
                f"(|has_mirror|) and ({exit.text_access_rule})"
                if exit.text_access_rule
                else "|has_mirror|"
            )
        # Track entrances from OW to UW
        if exit.connected_region:
            if exit.connected_region.name in ignored_tos:
                continue
            exit_dict = {
                "to": exit.connected_region.name,
                "type": exit.connected_region.type.name,
                "requirements": {"Open": "never", "Inverted": check_rule_keys(exit)},
            }
            all_rules.add(json.dumps(check_rule_keys(exit)))
            # Inverted only region
            if exit.name not in exits:
                if region.type in [RegionType.LightWorld, RegionType.DarkWorld]:
                    entrances.append(exit.name)
                    exits[exit.name] = exit_dict
            # Inverted has unique connected region from same exit
            elif exits[exit.name]["to"] != exit.connected_region.name:
                exits[exit.name + " (Inverted)"] = exit_dict
            else:
                exit_dict = exits[exit.name]
                exit_dict["requirements"]["Inverted"] = check_rule_keys(exit)
                all_rules.add(json.dumps(check_rule_keys(exit)))
    for location in region.locations:
        if location.name not in locations:
            locations[location.name] = {
                "requirements": {"Open": "never", "Inverted": check_rule_keys(location)}
            }
            all_rules.add(json.dumps(check_rule_keys(location)))
            if "Enemy" in location.name:
                locations[location.name]["type"] = location.note
        else:
            locations[location.name]["requirements"]["Inverted"] = check_rule_keys(
                location
            )
            all_rules.add(json.dumps(check_rule_keys(location)))

In [12]:
from DoorShuffle import default_small_key_doors, key_doors

all_small_key_doors = []
for dp in key_doors:
    all_small_key_doors.append(dp[0])
    all_small_key_doors.append(dp[1])

# 'Eastern Palace': [
#     ('Eastern Dark Square Key Door WN', 'Eastern Cannonball Ledge Key Door EN'),
#     'Eastern Darkness Up Stairs',
# ],
for dungeon, doors in default_small_key_doors.items():
    for door in doors:
        if type(door) == str:
            all_small_key_doors.append(door)
        else:
            for d in door:
                all_small_key_doors.append(d)


all_small_key_doors.remove("Eastern Courtyard N")
all_small_key_doors.remove("Eastern Big Key NE")
all_small_key_doors.remove("Hera Startile Corner NW")
all_small_key_doors.remove("Desert Wall Slide NW")



In [13]:
entrance_graph = json.loads(json.dumps(entrance_graph))  # Deep copy to remove references

In [15]:
never = json.dumps({
    "always": {
        "allOf": [
            "never"
        ]
    }
})
exempt = [
    "Ice Lake Iceberg Bomb Jump",
    "Ice Lake Northeast Pier Hop",
    "Old Man Cave Exit (West)",
    "Paradox Cave Push Block Reverse",
    "Paradox Cave Bomb Jump",
]

big_key_doors = [
    "Hyrule Dungeon Cellblock Door",
    "Eastern Courtyard N",
    "Eastern Big Key NE",
    "Desert Wall Slide NW",
    "Hera Startile Corner NW",
    "PoD Dark Alley NE",
    "Thieves BK Corner NE",
    "Thieves Blind's Cell Door",
    "Ice Crystal Right NE",
    "Mire BK Door Room N",
    "Mire Square Rail NW",
    "Mire Antechamber NW",
    "TR Dodgers NE",
    "TR Final Abyss NW",
    "GT Dash Hall NE",
    "GT Brightly Lit Hall NW",
]

def has_allOf_not_never(req):
    """Check if requirement has always.allOf that is not ['never']"""
    return (isinstance(req, dict) and 
            "always" in req and 
            "allOf" in req["always"] and 
            req["always"]["allOf"] != ['never'])

def has_anyOf(req):
    """Check if requirement has always.anyOf"""
    return (isinstance(req, dict) and 
            "always" in req and 
            "anyOf" in req["always"])

def add_key_requirement(req, key_type):
    """Add a key requirement to existing requirements, handling allOf and anyOf cases"""
    if has_allOf_not_never(req):
        req["always"]["allOf"].append(key_type)
    elif has_anyOf(req):
        # Wrap the anyOf in an allOf with the key requirement
        existing_anyOf = req["always"]["anyOf"]
        req["always"] = {
            "allOf": [
                {"anyOf": existing_anyOf},
                key_type
            ]
        }
    else:
        # No existing requirements or requirements are 'never', just set the key
        req.clear()
        req["always"] = {"allOf": [key_type]}

for e, ed in entrance_graph.items():
    for ex, exd in ed['exits'].items():
        # Check if all requirements are "never"
        if exd['requirements']['Open'] == json.loads(never) and exd['requirements']['Inverted'] == json.loads(never):
            print(f"{ex} to {exd['to']} is always never")

for e, ed in entrance_graph.items():
    for ex, exd in ed['exits'].items():
        if ex in big_key_doors:
            add_key_requirement(exd['requirements']['Open'], 'bigkey')
            add_key_requirement(exd['requirements']['Inverted'], 'bigkey')
        elif ex in all_small_key_doors:
            add_key_requirement(exd['requirements']['Open'], 'smallkey')
            add_key_requirement(exd['requirements']['Inverted'], 'smallkey')
        if ex in one_way_doors:
            exd['requirements']['Open'] = json.loads(never)
            exd['requirements']['Inverted'] = json.loads(never)

Ice Lake Iceberg Bomb Jump to Ice Lake Iceberg is always never
Ice Lake Northeast Pier Hop to Ice Lake Northeast Bank is always never
Old Man Cave Exit (West) to Mountain Pass Entry is always never
Paradox Cave Push Block Reverse to Paradox Cave Chest Area is always never
Paradox Cave Bomb Jump to Paradox Cave is always never
PoD Arena Right to Ranged Crystal to PoD Arena Right - Ranged Crystal is always never
PoD Arena Ledge to Ranged Crystal to PoD Arena Ledge - Ranged Crystal is always never
Mire Fishbone Blue Barrier Bypass to Mire South Fish is always never
GT Double Switch Entry to Ranged Switches to GT Double Switch Entry - Ranged Switches is always never
GT Double Switch Left to Exit Bypass to GT Double Switch Exit is always never
GT Double Switch Pot Corners to Ranged Switches to GT Double Switch Pot Corners - Ranged Switches is always never


In [16]:
def add_allOf_requirement(req_dict, requirement):
    if type(req_dict) == str:
        if req_dict == "never":
            req_dict = {}
        else:
            req_dict = {"always": {"allOf": [req_dict]}}
    if "always" not in req_dict:
        req_dict["always"] = {}
    if "allOf" not in req_dict["always"]:
        if "anyOf" in req_dict["always"]:
            # Convert anyOf to allOf with single anyOf inside
            req_dict["always"]["allOf"] = [{"anyOf": req_dict["always"]["anyOf"]}, {"allOf": [requirement]}]
            del req_dict["always"]["anyOf"]
            return
        else:
            req_dict["always"]["allOf"] = []
    if req_dict["always"]["allOf"] == ["never"]:
        req_dict["always"]["allOf"] = []
    if requirement not in req_dict["always"]["allOf"]:
        req_dict["always"]["allOf"].append(requirement)

In [17]:
for region, region_data in entrance_graph.items():
    for e in region_data['exits'].keys():
        if e in hasCrystalSwitch:
            entrance_graph[region]['locations']['Crystal_Switch'] = {
                "requirements": {
                    "Open": {
                        "always": {
                            "allOf": [
                                "canHitSwitch"
                            ]
                        }
                    },
                    "Inverted": {
                        "always": {
                            "allOf": [
                                "canHitSwitch"
                            ]
                        }
                    }
                }
            }
        if e in blueBarrier:
            print(f"Adding blueSwitchDown to {e} in {region}")
            open_reqs = entrance_graph[region]['exits'][e]['requirements']['Open']
            inv_reqs = entrance_graph[region]['exits'][e]['requirements']['Inverted']
            add_allOf_requirement(open_reqs, "blueSwitchDown")
            add_allOf_requirement(inv_reqs, "blueSwitchDown")
            print(entrance_graph['Sewers Behind Tapestry']['exits']['Sewers Behind Tapestry S']['requirements']['Open']['always']['allOf'])
        if e in orangeBarrier:
            print(f"Adding orangeSwitchDown to {e} in {region}")
            open_reqs = entrance_graph[region]['exits'][e]['requirements']['Open']
            inv_reqs = entrance_graph[region]['exits'][e]['requirements']['Inverted']
            add_allOf_requirement(open_reqs, "orangeSwitchDown")
            add_allOf_requirement(inv_reqs, "orangeSwitchDown")
            print(entrance_graph['Sewers Behind Tapestry']['exits']['Sewers Behind Tapestry S']['requirements']['Open']['always']['allOf'])


Adding blueSwitchDown to Hera Lobby to Front Barrier - Blue in Hera Lobby
['never']
Adding blueSwitchDown to Hera Front to Lobby Barrier - Blue in Hera Front
['never']
Adding blueSwitchDown to Hera Front to Down Stairs Barrier - Blue in Hera Front
['never']
Adding orangeSwitchDown to Hera Front to Up Stairs Barrier - Orange in Hera Front
['never']
Adding orangeSwitchDown to Hera Front to Back Barrier - Orange in Hera Front
['never']
Adding blueSwitchDown to Hera Front to Back Bypass in Hera Front
['never']
Adding blueSwitchDown to Hera Down Stairs to Front Barrier - Blue in Hera Down Stairs Landing
['never']
Adding orangeSwitchDown to Hera Up Stairs to Front Barrier - Orange in Hera Up Stairs Landing
['never']
Adding orangeSwitchDown to Hera Back to Front Barrier - Orange in Hera Back
['never']
Adding blueSwitchDown to Hera Tile Room EN in Hera Tile Room
['never']
Adding blueSwitchDown to Hera Tridorm WN in Hera Tridorm
['never']
Adding orangeSwitchDown to Hera Tridorm SE in Hera Trido

In [18]:
# Manual fixes go here:

# PoD Fix (logic bug)
entrance_graph["PoD Arena Bridge"]["exits"]["PoD Arena Bridge Drop Down"]["to"] = "PoD Arena Landing"

# Spike cave - complex
# set_rule(world.get_location('Spike Cave', player), lambda state:
#          state.has('Hammer', player) and state.can_lift_rocks(player) and
#          ((state.has('Cape', player) and state.can_extend_magic(player, 16, True)) or
#          (state.has('Cane of Byrna', player) and
#           (state.can_extend_magic(player, 12, True) or (state.world.can_take_damage and (state.has_Boots(player) or state.has_hearts(player, 4)))
#            )))
#          )
spike_requirements = {
    "always": {
        "allOf": [
            "hammer",
            "glove",
        ],
        "anyOf": [
            {
                "allOf": [
                    "cape",
                    "canExtendMagic|16",
                ]
            },
            {
                "allOf": [
                    "byrna",
                    {
                        "anyOf": [
                            "canExtendMagic|12",
                            {
                                "allOf": [
                                    "canTakeDamage",
                                    {
                                        "anyOf": [
                                            "boots",
                                            "hearts|4",
                                        ]
                                    }
                                ]
                            }
                        ]
                    }
                ]
            }
        ],
    }
}

for location in entrance_graph["Spike Cave"]["locations"]:
    entrance_graph["Spike Cave"]["locations"][location]["requirements"]["Open"] = spike_requirements
    entrance_graph["Spike Cave"]["locations"][location]["requirements"]["Inverted"] = spike_requirements


entrance_graph["Mire Area"]["exits"]["Misery Mire"]["requirements"] = {
    "Open": {"always": {"allOf": ["canOpenMM"]}},
    "Inverted": {"always": {"allOf": ["canOpenMM"]}},
}


# Swamp draining
# Swamp Hammer Switch
entrance_graph["Swamp Trench 1 Key Ledge"]["exits"]["Swamp Trench 1 Key Ledge Depart"][
    "requirements"
]["Open"]["always"]["allOf"] += ["canReach|Swamp Hammer Switch", "hammer"]

entrance_graph["Swamp Trench 1 Key Ledge"]["exits"]["Swamp Trench 1 Key Ledge Depart"][
    "requirements"
]["Inverted"]["always"]["allOf"] += ["canReach|Swamp Hammer Switch", "hammer"]

entrance_graph["Swamp Trench 1 Approach"]["exits"][
    "Swamp Trench 1 Approach Swim Depart"
]["requirements"]["Open"]["always"]["allOf"] += ["canReach|Swamp Hammer Switch", "hammer"]

entrance_graph["Swamp Trench 1 Approach"]["exits"][
    "Swamp Trench 1 Approach Swim Depart"
]["requirements"]["Inverted"]["always"]["allOf"] += ["canReach|Swamp Hammer Switch", "hammer"]


# Swamp draining
# Swamp Tranch 2
add_allOf_requirement(entrance_graph["Swamp Trench 2 Pots"]["exits"]["Swamp Trench 2 Pots Wet"][
    "requirements"
]["Open"], "canReach|Swamp Crystal Switch Inner")

add_allOf_requirement(entrance_graph["Swamp Trench 2 Pots"]["exits"]["Swamp Trench 2 Pots Wet"][
    "requirements"
]["Inverted"], "canReach|Swamp Crystal Switch Inner")



add_allOf_requirement(
    entrance_graph["Ice Jelly Key"]["exits"]["Ice Jelly Key Down Stairs"][
        "requirements"
    ]["Open"],
    "smallkey",
)

add_allOf_requirement(
    entrance_graph["Ice Jelly Key"]["exits"]["Ice Jelly Key Down Stairs"][
        "requirements"
    ]["Inverted"],
    "smallkey",
)


# Inverted region fixes
entrance_graph["Big Bomb Shop"]["exits"]["Big Bomb Shop Exit"] = {
    "to": "Big Bomb Shop Area",
    "type": "DarkWorld",
    "requirements": {"Open": {}, "Inverted": {}},
}

entrance_graph["Dark Sanctuary Hint"]["exits"]["Dark Sanctuary Hint Exit"] = {
    "to": "Dark Chapel Area",
    "type": "DarkWorld",
    "requirements": {"Open": {}, "Inverted": {}},
}

# Ice Palace logic
for i in ["Open", "Inverted"]:
    entrance_graph["Ice Crystal Left"]["exits"]["Ice Crystal Left Blue Barrier"][
        "requirements"
    ][i] = {"always": {"allOf": ["canHitSwitch", "canReach|Ice Refill"]}}

# One way trap doors
entrance_graph["GT Bob's Torch"]["exits"]["GT Torch SW"]["requirements"] = {
    "Open": {"always": {"allOf": ["never"]}},
    "Inverted": {"always": {"allOf": ["never"]}},
}

entrance_graph["Mire Wizzrobe Bypass"]["exits"]["Mire Wizzrobe Bypass WN"][
    "requirements"
] = {
    "Open": {"always": {"allOf": ["never"]}},
    "Inverted": {"always": {"allOf": ["never"]}},
}

entrance_graph["Ice Hookshot Ledge"]["exits"]["Ice Hookshot Ledge WN"][
    "requirements"
] = {
    "Open": {"always": {"allOf": ["never"]}},
    "Inverted": {"always": {"allOf": ["never"]}},
}

entrance_graph["GT Bob's Torch"]["exits"]["GT Torch SW"]["requirements"] = {
    "Open": {"always": {"allOf": ["never"]}},
    "Inverted": {"always": {"allOf": ["never"]}},
}

# Old man exit broken
entrance_graph["Old Man Cave Ledge"]["exits"]["Old Man Cave Exit (West)"]["requirements"]["Open"] = {}
entrance_graph["Old Man Cave Ledge"]["exits"]["Old Man Cave Exit (West)"]["requirements"]["Inverted"] = {}

In [19]:

# Inverted 1.0

# - Links House start now spawns in Big Bomb Shop - DONE
# Fix menu exit
entrance_graph["Menu"]["exits"]["Links House S&Q (Inverted)"]['requirements']['Inverted_1'] = "never"
entrance_graph["Menu"]["exits"]["Links House S&Q"]['requirements']['Inverted_1'] = {}

# Relink link's house 
entrance_graph["Links House Area"]["exits"]["Links House"]['requirements']['Inverted_1'] = "never"
entrance_graph["Links House"]["exits"]["Links House Exit"]['requirements']['Inverted_1'] = "never"
entrance_graph["Links House"]["exits"]["Links House Exit (Inverted_1)"] = {
    "to": "Big Bomb Shop Area",
    "type": "DarkWorld",
    "requirements": {
        "Open": "never",
        "Inverted": "never",
        "Inverted_1": {}

    }
}
entrance_graph["Links House Area"]["exits"]["Big Bomb Shop"] = {
    "to": "Big Bomb Shop",
    "type": "Cave",
    "requirements": {
        "Open": "never",
        "Inverted": "never",
        "Inverted_1": {}
    }
}

# Relink Big Bomb Shop
entrance_graph["Big Bomb Shop Area"]["exits"]["Big Bomb Shop"]['requirements']['Inverted_1'] = "never"
entrance_graph["Big Bomb Shop"]["exits"]["Big Bomb Shop Exit"]['requirements']['Inverted_1'] = "never"
entrance_graph["Big Bomb Shop"]["exits"]["Big Bomb Shop Exit (Inverted_1)"] = {
    "to": "Links House Area",
    "type": "LightWorld",
    "requirements": {
        "Open": "never",
        "Inverted": "never",
        "Inverted_1": {}
    }
}
entrance_graph["Big Bomb Shop Area"]["exits"]["Links House"] = {
    "to": "Links House",
    "type": "Cave",
    "requirements": {
        "Open": "never",
        "Inverted": "never",
        "Inverted_1": {}
    }
}


# - Mountain Cave S+Q now spawns in his usual Cave - DONE
entrance_graph["Menu"]["exits"]["Old Man S&Q"]['requirements']['Inverted_1'] = "never"
entrance_graph["Menu"]["exits"]["Old Man S&Q (Inverted_1)"] = {
    "to": "West Dark Death Mountain (Bottom)",
    "type": "DarkWorld",
    "requirements": {
        "Open": "never",
        "Inverted": "never",
        "Inverted_1": {
            "always": {
                "allOf": [
                    "canRescueOldMan"
                ]
            }
        }
    }
}

# - Flute Spot 1 is moved to Top of Dark Death Mountain - DONE
entrance_graph["Flute Sky"]["exits"]["Flute Spot 1 (Inverted)"]['requirements']['Inverted_1'] = "never"
entrance_graph["Flute Sky"]["exits"]["Flute Spot 1 (Inverted_1)"] = {
    "to": "West Dark Death Mountain (Bottom)",
    "type": "DarkWorld",
    "requirements": {
        "Open": "never",
        "Inverted": "never",
        "Inverted_1": {
            "always": {
                "allOf": [
                    "canFlute"
                ]
            }
        }
    }
}


# - Spiral and Mimic Cave are now bridged together - DONE
# - Just make all of these exits not possible in Inv1
for exit in entrance_graph["Spiral Mimic Ledge Extend"]["exits"]:
    entrance_graph["Spiral Mimic Ledge Extend"]["exits"][exit]['requirements']['Inverted_1'] = "never"

# Re-add the inverted dropdpown
entrance_graph["East Death Mountain (Top East)"]['exits']['Mimic Cave Drop (Inverted_1)'] = {
    "to": "Mimic Cave Ledge",
    "type": "LightWorld",
    "requirements": {
        "Open": "never",
        "Inverted": "never",
        "Inverted_1": {}
    }
}

# - Ladder is removed from West Dark Death Mountain - DONE
# Reverse way is just a dropdown and already exists
entrance_graph["West Dark Death Mountain (Bottom)"]['exits']['West Dark Death Mountain Ladder (Inverted_1)'] = {
    "to": "West Dark Death Mountain (Top)",
    "type": "DarkWorld",
    "requirements": {
        "Open": "never",
        "Inverted": "never",
        "Inverted_1": {}
    }
}


# - TR Peg Puzzle is restored but instead reveals a ladder
# - Not needed. The requirements are the same anyway (mitts + hammer)

# - Old Man Cave/Bumper Cave returned to vanilla
# Relink Bumper cave bottom
entrance_graph["Bumper Cave Entry"]["exits"]["Bumper Cave (Bottom)"]['requirements']['Inverted_1'] = "never"
entrance_graph["Bumper Cave Entry"]["exits"]["Bumper Cave (Bottom Inverted_1)"] = {
    "to": "Old Man Cave Ledge",
    "type": "Cave",
    "requirements": {
        "Open": "never",
        "Inverted": "never",
        "Inverted_1": {}
    }
}
entrance_graph["Old Man Cave Ledge"]["exits"]["Old Man Cave Exit (West)"]["requirements"]["Inverted_1"] = "never"
entrance_graph["Old Man Cave Ledge"]["exits"]["Old Man Cave Exit (West Inverted_1)"] = {
    "to": "Bumper Cave Entry",
    "type": "DarkWorld",
    "requirements": {
        "Open": "never",
        "Inverted": "never",
        "Inverted_1": {}
    }
}

# Relink DWWDM Fairy
entrance_graph["West Dark Death Mountain (Bottom)"]["exits"]["Dark Death Mountain Fairy"]['requirements']['Inverted_1'] = "never"
entrance_graph["West Dark Death Mountain (Bottom)"]["exits"]["Dark Death Mountain Fairy (Inverted_1)"] = {
    "to": "Old Man Cave (East)",
    "type": "Cave",
    "requirements": {
        "Open": "never",
        "Inverted": "never",
        "Inverted_1": {}
    }
}

entrance_graph["Old Man Cave (East)"]["exits"]["Old Man Cave Exit (East)"]["requirements"]["Inverted_1"] = "never"
entrance_graph["Old Man Cave (East)"]["exits"]["Old Man Cave Exit (East Inverted_1)"] = {
    "to": "West Dark Death Mountain (Bottom)",
    "type": "DarkWorld",
    "requirements": {
        "Open": "never",
        "Inverted": "never",
        "Inverted_1": {
            "always": {
                "allOf": [
                    "lantern"
                ]
            }
        }
    }
}

# Relink old man cave
entrance_graph["West Death Mountain (Bottom)"]["exits"]["Old Man Cave (East)"]['requirements']['Inverted_1'] = "never"
entrance_graph["West Death Mountain (Bottom)"]["exits"]["Old Man Cave (East Inverted_1)"] = {
    "to": "Death Mountain Return Cave (left)",
    "type": "Cave",
    "requirements": {
        "Open": "never",
        "Inverted": "never",
        "Inverted_1": {}
    }
}

entrance_graph["Death Mountain Return Cave (left)"]["exits"]["Death Mountain Return Cave Exit (West)"]['requirements']['Inverted_1'] = "never"
entrance_graph["Death Mountain Return Cave (left)"]["exits"]["Death Mountain Return Cave Exit (West Inverted_1)"] = {
    "to": "West Death Mountain (Bottom)",
    "type": "LightWorld",
    "requirements": {
        "Open": "never",
        "Inverted": "never",
        "Inverted_1": {}
    }
}

# Relink DMA cave
entrance_graph["Mountain Pass Entry"]["exits"]["Old Man Cave (West)"]['requirements']['Inverted_1'] = "never"
entrance_graph["Mountain Pass Entry"]["exits"]["Old Man Cave (West Inverted_1)"] = {
    "to": "Bumper Cave (bottom)",
    "type": "Cave",
    "requirements": {
        "Open": "never",
        "Inverted": "never",
        "Inverted_1": {}
    }
}
entrance_graph["Bumper Cave (bottom)"]["exits"]["Bumper Cave Exit (Bottom)"]['requirements']['Inverted_1'] = "never"
entrance_graph["Bumper Cave (bottom)"]["exits"]["Bumper Cave Exit (Inverted_1)"] = {
    "to": "Mountain Pass Entry",
    "type": "LightWorld",
    "requirements": {
        "Open": "never",
        "Inverted": "never",
        "Inverted_1": {}
    }
}

# Relink Bumper Cave top
entrance_graph["Bumper Cave Ledge"]["exits"]["Bumper Cave (Top)"]['requirements']['Inverted_1'] = "never"
entrance_graph["Bumper Cave Ledge"]["exits"]["Bumper Cave (Top Inverted_1)"] = {
    "to": "Dark Death Mountain Healer Fairy",
    "type": "Cave",
    "requirements": {
        "Open": "never",
        "Inverted": "never",
        "Inverted_1": {}
    }
}

# Relink DMA Cave lower
entrance_graph["Mountain Pass Ledge"]["exits"]["Death Mountain Return Cave (West)"]['requirements']['Inverted_1'] = "never"
entrance_graph["Mountain Pass Ledge"]["exits"]["Death Mountain Return Cave (West Inverted_1)"] = {
    "to": "Bumper Cave (top)",
    "type": "Cave",
    "requirements": {
        "Open": "never",
        "Inverted": "never",
        "Inverted_1": {}
    }
}

# Relink DMD cave right
entrance_graph["Bumper Cave (top)"]["exits"]["Bumper Cave Exit (Top)"]['requirements']['Inverted_1'] = "never"
entrance_graph["Bumper Cave (top)"]["exits"]["Bumper Cave Exit (Top Inverted_1)"] = {
    "to": "Mountain Pass Ledge",
    "type": "LightWorld",
    "requirements": {
        "Open": "never",
        "Inverted": "never",
        "Inverted_1": {}
    }
}


# - Ice Palace has been re-sealed to vanilla, portal moved to outer edge of moat
entrance_graph["Ice Lake Iceberg"]["exits"]["Ice Palace (Inverted)"] = {
    "to": "Ice Palace Area",
    "type": "DarkWorld",
    "requirements": {
        "Open": "never",
        "Inverted": "never",
        "Inverted_1": {}
    }
}

entrance_graph["Ice Palace Area"]["exits"]["Ice Lake Iceberg"] = {
    "to": "Ice Lake Iceberg",
    "type": "DarkWorld",
    "requirements": {
        "Open": "never",
        "Inverted": "never",
        "Inverted_1": {}
    }
}

entrance_graph["Ice Lake Water"]["exits"]["Ice Lake Iceberg"] = {
    "to": "Ice Lake Iceberg",
    "type": "DarkWorld",
    "requirements": {
        "Open": {
            "always": {
                "allOf": [
                    "moonpearl"
                    "flippers"
                ]
            }
        },
        "Inverted": {
            "always": {
                "allOf": [
                    "flippers"
                ]
            }
        },
        "Inverted_1":{
            "always": {
                "allOf": [
                    "flippers"
                ]
            }
        }
    }
}

entrance_graph["Ice Lake Iceberg"]["exits"]["Ice Lake Water"] = {
    "to": "Ice Lake Water",
    "type": "DarkWorld",
    "requirements": {
        "Open": {
            "always": {
                "allOf": [
                    "moonpearl"
                    "flippers"
                ]
            }
        },
        "Inverted": {
            "always": {
                "allOf": [
                    "flippers"
                ]
            }
        },
        "Inverted_1":{
            "always": {
                "allOf": [
                    "flippers"
                ]
            }
        }
    }
}

In [20]:

# Add boss kill requirements to Boss locations
for region_name, region_data in entrance_graph.items():
    for exit_name, exit_data in region_data["exits"].items():
        if exit_data["to"].endswith(" Boss Spoils"):
            add_allOf_requirement(
                exit_data["requirements"]["Open"], "canKillBoss"
            )
            add_allOf_requirement(
                exit_data["requirements"]["Inverted"], "canKillBoss"
            )

# TODO: Also split this into doors vs chest kill rooms
# TODO: Remember to detect only killable enemies for the logic
# TODO: Add the actual kill logic per enemy type
kill_rooms = [
    "Hyrule Dungeon Armory Main",
    "Hyrule Dungeon Armory Boomerang",
    "Desert Trap Room",
    "Desert Four Statues",
    "Hera Beetles",
    "Tower Gold Knights",
    # 'Tower Room 03', # Room is not, but chest is
    "Tower Red Spears",
    "Tower Red Guards",
    "PoD Turtle Party",
    # "Swamp Entrance", # Chest only
    "Thieves Trap",
    "PoD Mimics 1",
    "PoD Mimics 2",
    "Mire 2",
]


In [21]:
with open("all_rules_parsed.txt", "w") as f:
    for rule in sorted(all_rules):
        f.write(rule + "\n")

In [22]:
with open("logic_regions.ts", "w") as f:
    # // This file is auto-generated by generate_overworld_entrances.py
    # import type { OverworldRegionLogic } from "@/data/logic/logicTypes";

    # export const logic_regions: Record<string, OverworldRegionLogic> = {
    f.write("// This file is auto-generated by generate_overworld_entrances.py\n")
    f.write('import type { RegionLogic } from "@/data/logic/logicTypes";\n\n')
    f.write('export const logic_regions: Record<string, RegionLogic> = ')
    import json
    f.write(json.dumps(entrance_graph, indent=4))
    f.write(";\n")

In [23]:
with open("../alttptracker_react/src/data/logic/logic_regions.ts", "w") as f:
    # // This file is auto-generated by generate_overworld_entrances.py
    # import type { OverworldRegionLogic } from "@/data/logic/logicTypes";

    # export const logic_regions: Record<string, OverworldRegionLogic> = {
    f.write("// This file is auto-generated by generate_overworld_entrances.py\n")
    f.write('import type { RegionLogic } from "@/data/logic/logicTypes";\n\n')
    f.write('export const logic_regions: Record<string, RegionLogic> = ')
    import json
    f.write(json.dumps(entrance_graph, indent=4))
    f.write(";\n")